# Market Differentiated Bars

Calculate a fixed-width fractional difference of the observed AAPL dollar-bar log price, selecting the minimum tested order that passes the ADF 5% critical value.

## Process the Data

- Fixed-width fractional differencing is applied to log close prices.
- A 0.01 absolute-weight cutoff shortens the retained memory, candidate orders span 0.0 to 1.0 in 0.1 steps, and the smallest tested order whose ADF statistic passes the 5% critical value is selected; the coarse grid favors an auditable stationarity-versus-memory trade-off rather than claiming a globally optimal order.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_preprocessing.market_differentiated_bars import (
    fractional_difference_fixed_width,
    plot_min_ffd,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_bar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
fractional_path = feature_dir / f"aapl_dollar_bar_fractional_{period}.parquet"

if not fractional_path.is_file():
    bars = pd.read_parquet(dollar_bar_path, columns=["end", "close"])
    log_close = np.log(bars[["close"]].astype(float)).rename(
        columns={"close": "log_close"}
    )
    weight_cutoff = 0.01
    diagnostics = plot_min_ffd(
        log_close,
        weight_cutoff=weight_cutoff,
        differencing_orders=np.linspace(0.0, 1.0, 11),
    )
    stationary_orders = diagnostics.index[
        diagnostics["adf_statistic"] < diagnostics["critical_value_5pct"]
    ]
    if stationary_orders.empty:
        raise ValueError("No tested differencing order passed the ADF 5% critical value.")
    selected_order = float(stationary_orders.min())
    differentiated = fractional_difference_fixed_width(
        log_close,
        differencing_order=selected_order,
        weight_cutoff=weight_cutoff,
    )
    fractional_bars = bars.loc[differentiated.index, ["end"]].copy()
    fractional_bars["fractionally_differenced_log_close"] = differentiated["log_close"]
    fractional_bars.to_parquet(fractional_path, index=False)

fractional_bars = pd.read_parquet(fractional_path)
fractional_path

PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/market/features/aapl_dollar_bar_fractional_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- These checks verify timestamps, dtype, range, and distribution of the persisted fractionally differenced feature without recalculating it.
- Histogram bins and figure size affect only the visualization.

In [2]:
fractional_bars.head()

,end,fractionally_differenced_log_close
0,2025-01-02 14:34:00.996465+00:00,2.045494
1,2025-01-02 14:34:49.957222+00:00,2.044971
2,2025-01-02 14:35:39.368296+00:00,2.043638
3,2025-01-02 14:36:17.392377+00:00,2.044344
4,2025-01-02 14:36:40.115118+00:00,2.043984


In [3]:
fractional_bars.info()

<class 'pandas.DataFrame'>
RangeIndex: 80202 entries, 0 to 80201
Data columns (total 2 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   end                                 80202 non-null  datetime64[us, UTC]
 1   fractionally_differenced_log_close  80202 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(1)
memory usage: 1.2 MB


In [4]:
fractional_bars.dtypes.value_counts()

datetime64[us, UTC]    1
float64                1
Name: count, dtype: int64

In [5]:
fractional_bars[["fractionally_differenced_log_close"]].describe()

,fractionally_differenced_log_close
count,80202.000000
mean,2.023838
std,0.045243
min,1.889691
25%,1.986621
50%,2.024010
75%,2.062857
max,2.114264


In [6]:
fractional_bars[["fractionally_differenced_log_close"]].hist(
    bins=50, figsize=(6, 4)
)
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_82290/1202119065.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
